### Machine Performance Monitor

Supports monitoring a remote machine, a VM inside a remote machine, or the local machine.

* **Configuration**: Edit the settings in the first code cell.
* **Remote/Local**: Use `MONITOR_REMOTE` to switch between remote and local monitoring.
* **VM/Host**: Use `MONITOR_REMOTE_VM` to switch between monitoring the VM and the jump host.
* **SSH Config**: For remote monitoring, set up your SSH credentials in the `sshConfig` file.
* **Metrics**: Toggle the boolean flags (`MONITOR_CPU`, `MONITOR_MEM`, etc.) to enable or disable specific metrics.

In [ ]:
# ==================== Configuration ====================
MONITOR_REMOTE = False   # True to monitor remote, False to monitor local
MONITOR_REMOTE_VM = False    # For remote: True to monitor the VM, False for the jump host

INTERVAL = 5            # Sampling interval (seconds)
DURATION = 30          # Total duration (seconds), set to None for infinite execution

# --- Switches for metrics ---
MONITOR_CPU = True
MONITOR_MEM = True
MONITOR_DISK_USAGE = True
MONITOR_DISK_IO = False   # Requires iostat (sysstat package)
MONITOR_NETWORK = True
MONITOR_LOAD = True
MONITOR_PROCESSES = True

# --- Device names ---
NET_INTERFACE = 'eth0'    # e.g., eth0, ens33
DISK_DEVICE = 'sda'       # e.g., sda, vda

In [ ]:
import time
import matplotlib.pyplot as plt
from IPython.display import clear_output
from monitoring_logic import setup_client, MetricsCollector

# Pack configuration into a dictionary
config = {
    'MONITOR_REMOTE': MONITOR_REMOTE,
    'MONITOR_REMOTE_VM': MONITOR_REMOTE_VM,
    'INTERVAL': INTERVAL,
    'DURATION': DURATION,
    'MONITOR_CPU': MONITOR_CPU,
    'MONITOR_MEM': MONITOR_MEM,
    'MONITOR_DISK_USAGE': MONITOR_DISK_USAGE,
    'MONITOR_DISK_IO': MONITOR_DISK_IO,
    'MONITOR_NETWORK': MONITOR_NETWORK,
    'MONITOR_LOAD': MONITOR_LOAD,
    'MONITOR_PROCESSES': MONITOR_PROCESSES,
    'NET_INTERFACE': NET_INTERFACE,
    'DISK_DEVICE': DISK_DEVICE
}

def draw_plots(times, data, config, host_alias):
    clear_output(wait=True)
    
    enabled_metrics = [k for k, v in config.items() if k.startswith('MONITOR_') and v]
    n_plots = len(enabled_metrics)
    if n_plots == 0:
        print("No monitoring metrics enabled. Please modify the configuration.")
        return

    fig = plt.figure(figsize=(14, 2.5 * n_plots))
    gs = fig.add_gridspec(n_plots, 1, hspace=0.5)
    plot_idx = 0
    
    if config.get('MONITOR_CPU'):
        ax = fig.add_subplot(gs[plot_idx])
        ax.plot(times, data['cpu'], 'r-', linewidth=1.5)
        ax.set_ylabel('CPU (%)')
        ax.set_ylim(0, 100)
        ax.grid(True)
        ax.set_title('CPU Usage')
        plot_idx += 1
        
    if config.get('MONITOR_MEM'):
        ax = fig.add_subplot(gs[plot_idx])
        ax.plot(times, data['mem'], 'g-', linewidth=1.5)
        ax.set_ylabel('Memory (%)')
        ax.set_ylim(0, 100)
        ax.grid(True)
        ax.set_title('Memory Usage')
        plot_idx += 1

    if config.get('MONITOR_DISK_USAGE'):
        ax = fig.add_subplot(gs[plot_idx])
        ax.plot(times, data['disk_usage'], 'b-', linewidth=1.5)
        ax.set_ylabel('Disk Usage (%)')
        ax.set_ylim(0, 100)
        ax.grid(True)
        ax.set_title('Root Disk Usage')
        plot_idx += 1

    if config.get('MONITOR_DISK_IO'):
        ax = fig.add_subplot(gs[plot_idx])
        ax.plot(times, data['disk_io'], 'orange', linewidth=1.5)
        ax.set_ylabel('Disk I/O %util')
        ax.set_ylim(0, 100)
        ax.grid(True)
        ax.set_title(f'Disk I/O ({config["DISK_DEVICE"]})')
        plot_idx += 1

    if config.get('MONITOR_NETWORK') and len(data['net_sent']) > 0:
        ax = fig.add_subplot(gs[plot_idx])
        ax.plot(times, data['net_sent'], 'c-', label='Sent', linewidth=1.5)
        ax.plot(times, data['net_recv'], 'm-', label='Received', linewidth=1.5)
        ax.set_ylabel('Network (bytes/s)')
        ax.legend()
        ax.grid(True)
        ax.set_title(f'Network Traffic ({config["NET_INTERFACE"]})')
        plot_idx += 1

    if config.get('MONITOR_LOAD'):
        ax = fig.add_subplot(gs[plot_idx])
        ax.plot(times, data['load1'], label='1 min', linewidth=1.5)
        ax.plot(times, data['load5'], label='5 min', linewidth=1.5)
        ax.plot(times, data['load15'], label='15 min', linewidth=1.5)
        ax.set_ylabel('Load Average')
        ax.legend()
        ax.grid(True)
        ax.set_title('System Load')
        plot_idx += 1

    if config.get('MONITOR_PROCESSES'):
        ax = fig.add_subplot(gs[plot_idx])
        ax.plot(times, data['processes'], 'purple', linewidth=1.5)
        ax.set_ylabel('Process Count')
        ax.grid(True)
        ax.set_title('Total Processes')
        plot_idx += 1
    
    fig.supxlabel('Time (seconds)')
    fig.suptitle(f'Performance Monitor - {host_alias}', fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

In [ ]:
# ==================== Monitoring Loop ====================
client, HOST_ALIAS = setup_client(MONITOR_REMOTE, MONITOR_REMOTE_VM, ssh_config_path="sshConfig")
collector = MetricsCollector(client, config)

times = []
data = {
    'cpu': [], 'mem': [], 'disk_usage': [], 'disk_io': [],
    'net_sent': [], 'net_recv': [], 'load1': [], 'load5': [], 'load15': [],
    'processes': []
}

start_time = time.time()
print(f"Starting to monitor {HOST_ALIAS}, updating every {INTERVAL} seconds...")
print("Press the ■ stop button in the Jupyter toolbar to stop.\n")

try:
    while True:
        elapsed = time.time() - start_time
        if DURATION and elapsed >= DURATION:
            print("\nMonitoring duration reached, stopping...")
            break
        
        times.append(elapsed)
        collector.collect_all_metrics(data)
        draw_plots(times, data, config, HOST_ALIAS)
        
        time.sleep(INTERVAL)
except KeyboardInterrupt:
    print("\nMonitoring interrupted by user.")
except Exception as e:
    print(f"An error occurred: {e}")
finally:
    client.close()
    print("Monitoring finished, connection closed.")